<a href="https://colab.research.google.com/github/ariyulistanbul/Pulmo/blob/v2/noduledetection_sgr_distilv1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from google.colab import drive

# 1. Drive'ı Bağla
drive.mount('/content/drive', force_remount=True)

# --- AYARLAR ---
# Kendi Drive yollarına göre burayı düzenle
NPZ_DIR = '/content/drive/MyDrive/Bitirme/LUNA_Processed'
CANDIDATES_CSV_PATH = '/content/drive/MyDrive/Bitirme/candidates.csv' # LUNA16'dan indirdiğin candidates dosyası
PATCH_SIZE = 32 # Modele girecek 3D küpün boyutu (32x32x32)

class Luna16Dataset(Dataset):
    def __init__(self, csv_path, npz_dir, patch_size=32):
        print("Veri seti haritası yükleniyor...")
        self.df = pd.read_csv(csv_path)
        self.npz_dir = npz_dir
        self.patch_size = patch_size

        # Sadece Drive'ımızda .npz dosyası hazır olan hastaları filtreleyelim
        available_patients = {f.replace('.npz', '') for f in os.listdir(npz_dir)}
        self.df = self.df[self.df['seriesuid'].isin(available_patients)].reset_index(drop=True)

        # Sınıf dağılımını görelim
        nodule_count = len(self.df[self.df['class'] == 1])
        non_nodule_count = len(self.df[self.df['class'] == 0])
        print(f"✅ Eşleşen Hasta Sayısı: {len(available_patients)}")
        print(f"🔍 Toplam Çıkarılacak Yama: {len(self.df)} (Nodül: {nodule_count}, Nodül Değil: {non_nodule_count})")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p_id = row['seriesuid']
        label = torch.tensor(row['class'], dtype=torch.float32) # 1 = Nodül, 0 = Değil

        # 1. NPZ dosyasını yükle
        data = np.load(os.path.join(self.npz_dir, f"{p_id}.npz"))
        scan, origin, spacing = data['scan'], data['origin'], data['spacing']

        # 2. Dünya koordinatlarını Voxel koordinatlarına çevir
        world_coord = np.array([row['coordZ'], row['coordY'], row['coordX']])
        v_coord = np.round(np.abs(world_coord - origin) / spacing).astype(int)
        z, y, x = v_coord

        # 3. 3D Yama (Patch) Kesme ve Sınır Kontrolü
        half = self.patch_size // 2
        z_min, z_max = max(0, z - half), min(scan.shape[0], z + half)
        y_min, y_max = max(0, y - half), min(scan.shape[1], y + half)
        x_min, x_max = max(0, x - half), min(scan.shape[2], x + half)

        patch = scan[z_min:z_max, y_min:y_max, x_min:x_max]

        # Eğer nodül akciğerin çok köşesindeyse ve kesilen yama 32x32x32'den küçükse, sıfırlarla doldur (Padding)
        if patch.shape != (self.patch_size, self.patch_size, self.patch_size):
            pad_z = self.patch_size - patch.shape[0]
            pad_y = self.patch_size - patch.shape[1]
            pad_x = self.patch_size - patch.shape[2]
            patch = np.pad(patch, ((0, pad_z), (0, pad_y), (0, pad_x)), mode='constant', constant_values=-1000)

        # 4. Normalizasyon (HU değerlerini 0-1 arasına çekme)
        patch = np.clip(patch, -1000, 400) # Akciğer dokusu genelde bu aralıktadır
        patch = (patch + 1000) / 1400.0

        # PyTorch 3D CNN'ler giriş olarak (Kanal, Derinlik, Yükseklik, Genişlik) bekler
        patch_tensor = torch.tensor(patch, dtype=torch.float32).unsqueeze(0)

        return patch_tensor, label.unsqueeze(0)

# Veri setini başlatalım (Test etmek için)
# dataset = Luna16Dataset(CANDIDATES_CSV_PATH, NPZ_DIR)

Mounted at /content/drive


In [ ]:
# Sadece candidates.csv dosyasını Kaggle'dan çekip Drive'a atıyoruz
!kaggle datasets download -d vafaeii/luna16 -f candidates.csv -p /content/temp_csv --unzip
!cp /content/temp_csv/candidates.csv /content/drive/MyDrive/Bitirme/candidates.csv
print("✅ candidates.csv Drive'a eklendi!")

Dataset URL: https://www.kaggle.com/datasets/vafaeii/luna16
License(s): Attribution-NonCommercial 4.0 International (CC BY-NC 4.0)
100% 52.9M/52.9M [00:05<00:00, 10.5MB/s]

✅ candidates.csv Drive'a eklendi!


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class NoduleClassifier3D(nn.Module):
    def __init__(self):
        super(NoduleClassifier3D, self).__init__()
        # Giriş Boyutu: [Batch_Size, Kanal=1, Derinlik=32, Yükseklik=32, Genişlik=32]

        # 1. Konvolüsyon Bloğu
        self.conv1 = nn.Conv3d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool3d(kernel_size=2, stride=2) # Boyutu yarıya indirir (16x16x16)

        # 2. Konvolüsyon Bloğu
        self.conv2 = nn.Conv3d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool3d(kernel_size=2, stride=2) # Boyutu yarıya indirir (8x8x8)

        # 3. Konvolüsyon Bloğu
        self.conv3 = nn.Conv3d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool3d(kernel_size=2, stride=2) # Boyutu yarıya indirir (4x4x4)

        # Tam Bağlantılı (Fully Connected) Katmanlar
        # 64 kanal * 4 * 4 * 4 boyutundaki tensörü düzleştiriyoruz
        self.fc1 = nn.Linear(64 * 4 * 4 * 4, 128)
        self.dropout = nn.Dropout(0.5) # Aşırı öğrenmeyi (Overfitting) engellemek için

        # Çıkış Katmanı: Nodül mü (1) Değil mi (0)
        self.fc2 = nn.Linear(128, 1)
        self.sigmoid = nn.Sigmoid() # Çıktıyı 0 ile 1 arasında bir olasılığa çevirir

    def forward(self, x):
        # Özellik Çıkarımı (Feature Extraction)
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.pool3(F.relu(self.conv3(x)))

        # Düzleştirme (Flattening)
        x = x.view(-1, 64 * 4 * 4 * 4)

        # Sınıflandırma
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        # Eğer BCELoss kullanacaksak aktivasyonsuz raw logit dönebiliriz,
        # ama klasik kullanımda Sigmoid işimizi kolaylaştırır.
        return self.sigmoid(x)

# Modeli test edelim
model = NoduleClassifier3D()
print(model)
print("✅ 3D CNN Mimarisi Hazır!")

# Rastgele bir 3D tensör oluşturup modelin çalışıp çalışmadığını test edelim
dummy_input = torch.randn(1, 1, 32, 32, 32) # [Batch, Channel, D, H, W]
output = model(dummy_input)
print(f"Test Çıktısı (0-1 arası olasılık): {output.item():.4f}")

NoduleClassifier3D(
  (conv1): Conv3d(1, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  (pool1): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  (pool2): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  (pool3): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=4096, out_features=128, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=128, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)
✅ 3D CNN Mimarisi Hazır!
Test Çıktısı (0-1 arası olasılık): 0.5258


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm
import shutil
import time
from google.colab import drive

# 1. Zombi bağlantıyı baştan temizleyelim
print("🔌 Drive bağlantısı sıfırlanıyor...")
drive.mount('/content/drive', force_remount=True)

# --- 2. AYARLAR VE YOLLAR ---
CANDIDATES_CSV_PATH = '/content/drive/MyDrive/Bitirme/candidates.csv'
DRIVE_NPZ_DIR = '/content/drive/MyDrive/Bitirme/LUNA_Processed'
LOCAL_TEMP_DIR = '/content/temp_npz' # Geçici güvenli alan
TENSOR_CACHE_PATH = '/content/drive/MyDrive/Bitirme/luna16_cached_tensors.pt'

os.makedirs(LOCAL_TEMP_DIR, exist_ok=True)

# --- 3. AKILLI YÜKLEME ---
if os.path.exists(TENSOR_CACHE_PATH):
    print(f"📦 Zaten işlenmiş veri bulundu! Drive'dan yükleniyor...")
    cached_data = torch.load(TENSOR_CACHE_PATH)
    X_tensor, y_tensor = cached_data['X'], cached_data['y']
    print(f"✅ Hazır! Toplam {len(X_tensor)} yama bellekte.")
else:
    print("🚀 Veriler sıfırdan kesilecek. ÖLÜMSÜZ BAĞLANTI SİSTEMİ DEVREDE...")

    df_all = pd.read_csv(CANDIDATES_CSV_PATH)
    available_patients = {f.replace('.npz', '') for f in os.listdir(DRIVE_NPZ_DIR)}
    df_all = df_all[df_all['seriesuid'].isin(available_patients)].reset_index(drop=True)

    df_nodules = df_all[df_all['class'] == 1]
    df_non_nodules = df_all[df_all['class'] == 0]

    TARGET_COUNT = len(df_nodules) * 3
    TARGET_COUNT = min(TARGET_COUNT, len(df_non_nodules))

    df_non_nodules_sampled = df_non_nodules.sample(n=TARGET_COUNT, random_state=42)
    df_balanced = pd.concat([df_nodules, df_non_nodules_sampled]).sample(frac=1, random_state=42).reset_index(drop=True)

    grouped = df_balanced.groupby('seriesuid')
    X_list = []
    y_list = []

    print("🚀 Yamalar RAM'e alınıyor. Drive koparsa kod otomatik müdahale edecek...")

    for p_id, group in tqdm(grouped, desc="Hastalar İşleniyor"):
        # Lanetli dosya koruması
        if p_id == "1.3.6.1.4.1.14519.5.2.1.6279.6001.101228986346984399347858840086":
            continue

        drive_path = os.path.join(DRIVE_NPZ_DIR, f"{p_id}.npz")
        local_path = os.path.join(LOCAL_TEMP_DIR, f"{p_id}.npz")

        # 🛡️ OTOMATİK TAMİRCİ (KOPMAYA KARŞI) 🛡️
        kopyalama_basarili = False
        while not kopyalama_basarili:
            try:
                if not os.path.exists(drive_path):
                    break # Dosya harbiden yoksa diğer hastaya geç

                shutil.copy(drive_path, local_path)
                kopyalama_basarili = True # Hatasız çalıştıysa döngüden çık

            except OSError as e:
                # Eğer bağlantı koptu hatası gelirse:
                if "Transport endpoint is not connected" in str(e) or getattr(e, 'errno', 0) == 107:
                    print(f"\n🛑 {p_id} okunurken Drive koptu! Kod kendi kendine yeniden takıyor...")
                    drive.mount('/content/drive', force_remount=True)
                    time.sleep(3) # Sistemin kendine gelmesini bekle ve BAŞA DÖN (while)
                else:
                    break # Başka bir hataysa uğraşma, diğer hastaya geç

        # Eğer defalarca denemeye rağmen kopyalanamadıysa es geç
        if not kopyalama_basarili:
            continue

        # --- YEREL DİSKTEN GÜVENLE OKUMA ---
        try:
            data = np.load(local_path)
            scan, origin, spacing = data['scan'], data['origin'], data['spacing']
            os.remove(local_path) # Çöp bırakma
        except Exception:
            if os.path.exists(local_path): os.remove(local_path)
            continue

        # --- YAMA KESME ---
        for _, row in group.iterrows():
            label = torch.tensor(row['class'], dtype=torch.float32)
            world_coord = np.array([row['coordZ'], row['coordY'], row['coordX']])

            v_coord = np.round(np.abs(world_coord - origin) / spacing).astype(int)
            z, y, x = v_coord

            half = 16
            z_min, z_max = max(0, z - half), min(scan.shape[0], z + half)
            y_min, y_max = max(0, y - half), min(scan.shape[1], y + half)
            x_min, x_max = max(0, x - half), min(scan.shape[2], x + half)

            patch = scan[z_min:z_max, y_min:y_max, x_min:x_max]

            if patch.shape != (32, 32, 32):
                pad_z, pad_y, pad_x = 32 - patch.shape[0], 32 - patch.shape[1], 32 - patch.shape[2]
                patch = np.pad(patch, ((0, pad_z), (0, pad_y), (0, pad_x)), mode='constant', constant_values=-1000)

            patch = np.clip(patch, -1000, 400)
            patch = (patch + 1000) / 1400.0

            X_list.append(torch.tensor(patch, dtype=torch.float32).unsqueeze(0))
            y_list.append(label.unsqueeze(0))

    print("\n🔥 BÜYÜK ZAFER! Veriler RAM'de! Tensörler oluşturuluyor...")
    X_tensor = torch.stack(X_list)
    y_tensor = torch.stack(y_list)

    # --- 4. GÜVENLİ KAYIT ---
    print(f"🔄 Işık hızında çıkarılan tensörler Drive'a yedekleniyor...")
    try:
        # Kopma ihtimaline karşı son bir remount
        drive.mount('/content/drive', force_remount=True)
        torch.save({'X': X_tensor, 'y': y_tensor}, TENSOR_CACHE_PATH)
        print(f"✅ Başarıyla Drive'a Kaydedildi! Toplam {len(X_tensor)} yama hazır.")
    except Exception as e:
        print(f"⚠️ Drive'a kopya atılamadı ama veriler RAM'de hazır, eğitime devam edilebilir. Hata: {e}")

🔌 Drive bağlantısı sıfırlanıyor...
Mounted at /content/drive
📦 Zaten işlenmiş veri bulundu! Drive'dan yükleniyor...
✅ Hazır! Toplam 5384 yama bellekte.


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# =====================================================================
# 🧠 MİMARİ: SEGRESNET (Sınıflandırma İçin Modifiye Edilmiş)
# =====================================================================
class SegResBlock(nn.Module):
    """Orijinal SegResNet Blok Mimarisi (GroupNorm + ReLU + Conv)"""
    def __init__(self, channels):
        super().__init__()
        # SegResNet'in alametifarikası Batch/Group Normalization kullanmasıdır
        self.norm1 = nn.GroupNorm(8, channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv3d(channels, channels, kernel_size=3, padding=1, bias=False)

        self.norm2 = nn.GroupNorm(8, channels)
        self.conv2 = nn.Conv3d(channels, channels, kernel_size=3, padding=1, bias=False)

    def forward(self, x):
        shortcut = x
        x = self.relu(self.norm1(x))
        x = self.conv1(x)
        x = self.relu(self.norm2(x))
        x = self.conv2(x)
        return x + shortcut # Skip Connection (Atlama Bağlantısı)

class SegResNet(nn.Module):
    def __init__(self, in_channels=1, init_filters=16):
        super().__init__()
        # 1. Başlangıç Konvolüsyonu
        self.convInit = nn.Conv3d(in_channels, init_filters, kernel_size=3, padding=1, bias=False)

        # 2. SegResNet Kodlayıcı (Encoder) Aşamaları
        self.down1 = nn.Conv3d(init_filters, init_filters*2, kernel_size=3, stride=2, padding=1, bias=False)
        self.block1 = SegResBlock(init_filters*2)

        self.down2 = nn.Conv3d(init_filters*2, init_filters*4, kernel_size=3, stride=2, padding=1, bias=False)
        self.block2 = SegResBlock(init_filters*4)

        self.down3 = nn.Conv3d(init_filters*4, init_filters*8, kernel_size=3, stride=2, padding=1, bias=False)
        self.block3 = SegResBlock(init_filters*8)

        # 3. Sınıflandırma Kafası (Orijinaldeki Decoder yerine)
        self.pool = nn.AdaptiveAvgPool3d((1, 1, 1))
        self.fc = nn.Linear(init_filters*8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Özellik Çıkarımı
        x = self.convInit(x)
        x = self.block1(self.down1(x))
        x = self.block2(self.down2(x))
        x = self.block3(self.down3(x))

        # Karar Verme (0 ile 1 arası olasılık)
        x = self.pool(x)
        x = x.view(x.size(0), -1) # Tensörü düzleştir
        x = self.fc(x)
        return self.sigmoid(x)


# =====================================================================
# 🚀 VERİ HAZIRLIĞI VE EĞİTİM BORU HATTI
# =====================================================================
print("🔪 RAM'deki veriler Eğitim (%80) ve Test (%20) olarak ayrılıyor...")

indices = list(range(len(X_tensor)))
labels = y_tensor.numpy()

train_idx, test_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=labels
)

X_train, y_train = X_tensor[train_idx], y_tensor[train_idx]
X_test, y_test = X_tensor[test_idx], y_tensor[test_idx]

print(f"📚 Eğitim Seti (Train): {len(X_train)} yama")
print(f"🎯 Test Seti (Validation): {len(X_test)} yama")

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# --- MODEL VE OPTİMİZASYON ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SegResNet().to(device) # SEGRESNET BURADA DEVREDE
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.BCELoss()

# --- CHECKPOINT MEKANİZMASI ---
EPOCHS = 10
CHECKPOINT_PATH = '/content/drive/MyDrive/Bitirme/luna16_segresnet_checkpoint.pth'
start_epoch = 0

if os.path.exists(CHECKPOINT_PATH):
    print(f"🔍 Checkpoint bulundu! Yükleniyor: {CHECKPOINT_PATH}")
    checkpoint = torch.load(CHECKPOINT_PATH)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch']
    print(f"✅ Kaldığımız yerden, {start_epoch + 1}. Epoch'tan devam ediliyor...")
else:
    print(f"🚀 Checkpoint bulunamadı. SegResNet Eğitimi {device} üzerinde sıfırdan başlıyor...")

# --- EĞİTİM VE TEST DÖNGÜSÜ ---
for epoch in range(start_epoch, EPOCHS):

    # ================= EĞİTİM FAZI =================
    model.train()
    running_train_loss = 0.0
    correct_train = 0
    total_train = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Eğitim]")

    for inputs, targets in progress_bar:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_fn(outputs, targets)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

        predicted = (outputs > 0.5).float()
        total_train += targets.size(0)
        correct_train += (predicted == targets).sum().item()

        progress_bar.set_postfix({'Loss': f"{loss.item():.4f}"})

    train_epoch_loss = running_train_loss / len(train_loader)
    train_epoch_acc = 100. * correct_train / total_train

    # ================= TEST FAZI =================
    model.eval()
    running_test_loss = 0.0
    correct_test = 0
    total_test = 0

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = loss_fn(outputs, targets)

            running_test_loss += loss.item()
            predicted = (outputs > 0.5).float()
            total_test += targets.size(0)
            correct_test += (predicted == targets).sum().item()

    test_epoch_loss = running_test_loss / len(test_loader)
    test_epoch_acc = 100. * correct_test / total_test

    # --- EPOCH ÖZETİ YAZDIRMA ---
    print(f"\n📊 Epoch [{epoch+1}/{EPOCHS}] Özeti:")
    print(f"   🔹 EĞİTİM -> Loss: {train_epoch_loss:.4f} | Accuracy: %{train_epoch_acc:.2f}")
    print(f"   🔸 TEST   -> Loss: {test_epoch_loss:.4f} | Accuracy: %{test_epoch_acc:.2f}")

    # --- CHECKPOINT KAYIT ---
    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': train_epoch_loss,
        'test_acc': test_epoch_acc
    }, CHECKPOINT_PATH)

    print(f"🔒 Epoch {epoch+1} güvenliğe alındı! Checkpoint Drive'a yazıldı.\n")

print("🎉 Tüm eğitim süreci başarıyla ve güvenle tamamlandı!")

🔪 RAM'deki veriler Eğitim (%80) ve Test (%20) olarak ayrılıyor...
📚 Eğitim Seti (Train): 4307 yama
🎯 Test Seti (Validation): 1077 yama
🔍 Checkpoint bulundu! Yükleniyor: /content/drive/MyDrive/Bitirme/luna16_segresnet_checkpoint.pth
✅ Kaldığımız yerden, 11. Epoch'tan devam ediliyor...
🎉 Tüm eğitim süreci başarıyla ve güvenle tamamlandı!


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

# --- 1. ÖĞRETMEN (TEACHER) MODELİN HAZIRLANMASI ---
print("👨‍🏫 Öğretmen Model (3D) hazırlanıyor...")
# Hafızadaki mevcut 'model' nesnemizi Öğretmen yapıyoruz
teacher_model = model.to(device)
teacher_model.eval() # Öğretmen artık öğrenmeyecek, sadece bildiklerini aktaracak

# Öğretmenin ağırlıklarını donduruyoruz (Boşuna işlemci/GPU harcamayalım)
for param in teacher_model.parameters():
    param.requires_grad = False

# --- 2. ÖĞRENCİ (STUDENT) MODELİN OLUŞTURULMASI (2.5D CNN) ---
print("👨‍🎓 Öğrenci Model (2.5D) doğuyor...")
class StudentCNN25D(nn.Module):
    def __init__(self):
        super(StudentCNN25D, self).__init__()
        # Giriş: 3 Kanal (Axial, Coronal, Sagittal kesitler), Boyut: 32x32
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2) # 16x16

        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2) # 8x8

        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(2, 2) # 4x4

        self.fc1 = nn.Linear(64 * 4 * 4, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.pool1(torch.relu(self.conv1(x)))
        x = self.pool2(torch.relu(self.conv2(x)))
        x = self.pool3(torch.relu(self.conv3(x)))

        x = x.view(-1, 64 * 4 * 4) # Düzleştir
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return self.sigmoid(x)

student_model = StudentCNN25D().to(device)
optimizer_s = optim.Adam(student_model.parameters(), lr=0.001)

# Damıtma (Distillation) Ayarları
alpha = 0.5 # %50 Öğretmenin Sözü, %50 Gerçek Etiket
bce_loss = nn.BCELoss() # Hem gerçek etiket hem de öğretmen tahmini için kullanacağız

# --- 3. BİLGİ DAMITMA (KNOWLEDGE DISTILLATION) DÖNGÜSÜ ---
EPOCHS = 10
print("🚀 Usta-Çırak (Distillation) Eğitimi Başlıyor!\n")

for epoch in range(EPOCHS):
    student_model.train()
    running_loss = 0.0
    correct_student = 0
    total = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Damıtma]")

    for inputs_3d, labels in progress_bar:
        inputs_3d, labels = inputs_3d.to(device), labels.to(device)

        # ✂️ ANLIK 2.5D KESİM İŞLEMİ (RAM Tasarrufu)
        # 3D Boyut: [Batch, 1, 32, 32, 32] -> Z, Y ve X eksenlerinin tam ortası (16. indeks)
        slice_z = inputs_3d[:, 0, 16, :, :] # Axial
        slice_y = inputs_3d[:, 0, :, 16, :] # Coronal
        slice_x = inputs_3d[:, 0, :, :, 16] # Sagittal

        # 3 dilimi üst üste koyarak 3 kanallı 2D resim yapıyoruz: [Batch, 3, 32, 32]
        inputs_25d = torch.stack((slice_z, slice_y, slice_x), dim=1)

        # 🧠 1. ÖĞRETMENİN FİKRİNİ AL
        with torch.no_grad():
            teacher_probs = teacher_model(inputs_3d) # Öğretmen 3D küpe bakar

        # 👶 2. ÖĞRENCİNİN TAHMİNİ
        optimizer_s.zero_grad()
        student_probs = student_model(inputs_25d) # Öğrenci 2.5D resme bakar

        # ⚖️ 3. DAMITMA KAYBI (DISTILLATION LOSS) HESAPLAMA
        # Öğrencinin gerçek cevaptan ne kadar saptığı
        loss_true = bce_loss(student_probs, labels)
        # Öğrencinin öğretmenden ne kadar saptığı (Karanlık Bilgi)
        loss_teacher = bce_loss(student_probs, teacher_probs)

        # İkisini harmanla
        loss = (alpha * loss_true) + ((1 - alpha) * loss_teacher)

        # Öğren (Ağırlıkları güncelle)
        loss.backward()
        optimizer_s.step()

        running_loss += loss.item()

        # Öğrencinin doğruluğunu hesapla (0.5 barajı)
        predicted = (student_probs > 0.5).float()
        total += labels.size(0)
        correct_student += (predicted == labels).sum().item()

        progress_bar.set_postfix({'Damıtma Loss': f"{loss.item():.4f}"})

    epoch_acc = 100. * correct_student / total
    print(f"📊 Öğrenci Eğitim Doğruluğu: %{epoch_acc:.2f} | Ortalama Kayıp: {running_loss/len(train_loader):.4f}")

    # ================= ÖĞRENCİ TEST FAZI =================
    student_model.eval()
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for inputs_3d, labels in test_loader:
            inputs_3d, labels = inputs_3d.to(device), labels.to(device)

            slice_z = inputs_3d[:, 0, 16, :, :]
            slice_y = inputs_3d[:, 0, :, 16, :]
            slice_x = inputs_3d[:, 0, :, :, 16]
            inputs_25d = torch.stack((slice_z, slice_y, slice_x), dim=1)

            outputs = student_model(inputs_25d)
            predicted = (outputs > 0.5).float()
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()

    print(f"   🔸 ÖĞRENCİ TEST ACCURACY (2.5D): %{100. * test_correct / test_total:.2f}\n")

print("🎉 2.5D Bilgi Damıtma Başarıyla Tamamlandı!")

👨‍🏫 Öğretmen Model (3D) hazırlanıyor...
👨‍🎓 Öğrenci Model (2.5D) doğuyor...
🚀 Usta-Çırak (Distillation) Eğitimi Başlıyor!



Epoch 1/10 [Damıtma]: 100%|██████████| 68/68 [00:02<00:00, 25.29it/s, Damıtma Loss=0.5463]


📊 Öğrenci Eğitim Doğruluğu: %74.09 | Ortalama Kayıp: 0.5801
   🔸 ÖĞRENCİ TEST ACCURACY (2.5D): %75.02



Epoch 2/10 [Damıtma]: 100%|██████████| 68/68 [00:00<00:00, 73.32it/s, Damıtma Loss=0.2805]


📊 Öğrenci Eğitim Doğruluğu: %78.69 | Ortalama Kayıp: 0.4651
   🔸 ÖĞRENCİ TEST ACCURACY (2.5D): %78.18



...

📊 Öğrenci Eğitim Doğruluğu: %88.90 | Ortalama Kayıp: 0.3121
   🔸 ÖĞRENCİ TEST ACCURACY (2.5D): %87.56



Epoch 9/10 [Damıtma]: 100%|██████████| 68/68 [00:00<00:00, 73.27it/s, Damıtma Loss=0.3308]


📊 Öğrenci Eğitim Doğruluğu: %89.20 | Ortalama Kayıp: 0.3089
   🔸 ÖĞRENCİ TEST ACCURACY (2.5D): %86.91



Epoch 10/10 [Damıtma]: 100%|██████████| 68/68 [00:00<00:00, 75.11it/s, Damıtma Loss=0.2552]


📊 Öğrenci Eğitim Doğruluğu: %90.41 | Ortalama Kayıp: 0.2821
   🔸 ÖĞRENCİ TEST ACCURACY (2.5D): %87.28

🎉 2.5D Bilgi Damıtma Başarıyla Tamamlandı!
